# Project 9 — Sports Vision System (detect -> track -> pitch coords)
```
synthetic pitch video -> color detector -> centroid tracker -> homography -> meters/heatmap
```
**Goal:** a mini end-to-end sports-analytics pipeline: from pixels to "who ran how far".
**Objectives:** (1) render a deterministic multi-player clip; (2) detect players classically
(color masks + connected components); (3) track with greedy centroid matching incl. occlusion
coasting; (4) solve pitch homography with DLT; (5) report distances/speeds + heatmap.

In [ ]:
import os # need os to create results/
import random # need random to seed the stdlib RNG
import time # need time to time detection/tracking
import csv # need csv to save tracks and summary tables
from collections import deque # need deque for BFS connected-component labeling
import numpy as np # need numpy for rendering, geometry, and tracking math
import matplotlib # need matplotlib for plots
matplotlib.use('Agg') # headless backend so nbconvert execute works
import matplotlib.pyplot as plt # need pyplot for frames/trajectories/heatmap
import matplotlib.patches as patches # need patches for box overlays
SEED = 9 # fixed seed: same clip every run
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and plots
try: # YOLO is the real-world detector for this pipeline stage
    import ultralytics # optional: only to report availability
    print('ultralytics', ultralytics.__version__, '(not needed: classical detector below)') # log version
except Exception: # not installed -> that is fine and expected here
    print('ultralytics not installed: using from-scratch color detector (offline-safe)') # log fallback

In [ ]:
W, H = 160, 120 # frame size: small keeps everything fast
NF = 48 # number of frames (~5s at 10fps)
FPS = 10.0 # assumed frame rate converts per-frame motion to m/s
PITCH_W, PITCH_H = 40.0, 30.0 # pitch size in meters (small-sided pitch)
GREEN = np.array([20, 140, 20], dtype=np.uint8) # pitch green
LINE = 220 # off-white line gray (ball is pure 255 so it stays separable)
RED = np.array([255, 30, 30], dtype=np.uint8) # team A color
BLUE = np.array([30, 30, 255], dtype=np.uint8) # team B color
IN = (12, 12, W-12, H-12) # inner pitch rect: (x0,y0,x1,y1) where lines are drawn
def disc(yy, xx, cy, cx, r): # boolean disc mask for drawing circles/lines
    return (yy-cy)**2 + (xx-cx)**2 <= r*r # circle equation on the pixel grid
Yg, Xg = np.mgrid[0:H, 0:W] # pixel coordinate grids reused by every frame
def draw_pitch(): # paint the empty pitch: green + white markings
    img = np.zeros((H, W, 3), dtype=np.uint8) + GREEN # flat green base
    x0, y0, x1, y1 = IN # unpack the inner rect
    img[y0:y0+2, x0:x1] = LINE # top touchline
    img[y1-2:y1, x0:x1] = LINE # bottom touchline
    img[y0:y1, x0:x0+2] = LINE # left touchline
    img[y0:y1, x1-2:x1] = LINE # right touchline
    img[y0:y1, W//2-1:W//2+1] = LINE # halfway line
    img[disc(Yg, Xg, H//2, W//2, 12) & ~disc(Yg, Xg, H//2, W//2, 10)] = LINE # center-circle ring
    return img # return the background frame
BG = draw_pitch() # render once, reuse for all frames
NPA, NPB = 4, 4 # 4 players per team keeps association readable
gen = np.random.default_rng(SEED) # dedicated RNG for the clip
PX = gen.uniform(20, W-20, size=NPA+NPB+1) # random initial x for 8 players + ball
PY = gen.uniform(20, H-20, size=NPA+NPB+1) # random initial y
VX = gen.uniform(-2.2, 2.2, size=NPA+NPB+1) # random x velocity (px/frame)
VY = gen.uniform(-1.8, 1.8, size=NPA+NPB+1) # random y velocity (px/frame)
GROUPS = ['A']*NPA + ['B']*NPB + ['ball'] # identity of each entity
RAD = [4]* (NPA+NPB) + [2] # player radius 4px, ball radius 2px
frames, GT = [], [] # frames holds images, GT holds per-frame entity positions
for t in range(NF): # simulate the clip frame by frame
    img = BG.copy() # start from the clean pitch
    occ = gen.random(NPA+NPB+1) < 0.06 # 6% occlusion: entity hidden this frame
    occ[-1] = gen.random() < 0.03 # ball occluded less often (smaller, informative)
    pos = [] # GT positions for this frame: (x, y, group, visible)
    for i in range(NPA+NPB+1): # move then draw each entity
        PX[i] += VX[i] # integrate x
        PY[i] += VY[i] # integrate y
        if PX[i] < 16 or PX[i] > W-16: # bounce off the side boards
            VX[i] *= -1 # flip x velocity
            PX[i] = float(np.clip(PX[i], 16, W-16)) # clamp back inside
        if PY[i] < 16 or PY[i] > H-16: # bounce off top/bottom
            VY[i] *= -1 # flip y velocity
            PY[i] = float(np.clip(PY[i], 16, H-16)) # clamp back inside
        if not occ[i]: # only draw when not occluded
            col = RED if GROUPS[i] == 'A' else (BLUE if GROUPS[i] == 'B' else np.array([255, 255, 255], dtype=np.uint8)) # pick entity color
            img[disc(Yg, Xg, PY[i], PX[i], RAD[i])] = col # stamp the disc
        pos.append((float(PX[i]), float(PY[i]), GROUPS[i], not occ[i])) # record GT incl. hidden ones
    frames.append(img) # store the frame
    GT.append(pos) # store GT for scoring
print(f'clip: {NF} frames {W}x{H}, entities={NPA+NPB+1}') # log clip specs

In [ ]:
def label_components(mask): # BFS connected-component labeling, from scratch
    lab = np.zeros_like(mask, dtype=np.int32) # 0 = background, >0 = component id
    cur = 0 # component counter
    comps = [] # pixel lists per component
    for y in range(mask.shape[0]): # scan every row
        for x in range(mask.shape[1]): # scan every column
            if mask[y, x] and lab[y, x] == 0: # unclaimed foreground pixel -> new blob
                cur += 1 # new component id
                q = deque([(y, x)]) # BFS queue starts here
                lab[y, x] = cur # claim the seed pixel
                px = [] # pixels of this blob
                while q: # flood fill
                    cy, cx = q.popleft() # pop next pixel
                    px.append((cy, cx)) # record it
                    for dy in (-1, 0, 1): # 8-neighbourhood rows
                        for dx in (-1, 0, 1): # 8-neighbourhood cols
                            ny, nx = cy+dy, cx+dx # neighbour coords
                            if 0 <= ny < mask.shape[0] and 0 <= nx < mask.shape[1]: # inside image
                                if mask[ny, nx] and lab[ny, nx] == 0: # unclaimed foreground
                                    lab[ny, nx] = cur # claim it
                                    q.append((ny, nx)) # enqueue it
                comps.append(px) # store this blob
    return comps # list of pixel lists
def detect_frame(img): # color-threshold detector for one frame
    f = img.astype(np.int16) # widen dtype so differences cannot overflow
    mA = (f[:, :, 0] > 180) & (f[:, :, 1] < 120) & (f[:, :, 2] < 120) # red-player mask
    mB = (f[:, :, 2] > 180) & (f[:, :, 0] < 120) & (f[:, :, 1] < 120) # blue-player mask
    mBa = (f[:, :, 0] > 245) & (f[:, :, 1] > 245) & (f[:, :, 2] > 245) # ball mask (lines are 220: excluded)
    dets = [] # (cx, cy, group, x0, y0, x1, y1) detections
    for mask, g, mn in [(mA, 'A', 12), (mB, 'B', 12), (mBa, 'ball', 4)]: # per-group masks + min blob size
        for px in label_components(mask): # each blob is one detection
            if len(px) < mn: # ignore speckle noise
                continue # too small to be real
            ys = [p[0] for p in px] # blob row coords
            xs = [p[1] for p in px] # blob col coords
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys) # bounding box
            dets.append(((x0+x1)/2, (y0+y1)/2, g, x0, y0, x1, y1)) # centroid + group + box
    return dets # all detections in this frame
t0 = time.time() # start detection timing
ALLD = [detect_frame(f) for f in frames] # detect every frame
dt_det = (time.time()-t0)/NF*1000 # ms per frame
print(f'detect: {sum(len(d) for d in ALLD)/NF:.2f} dets/frame, {dt_det:.2f} ms/frame') # speed + density
hits, vis = 0, 0 # recall counters
errs = [] # centroid error list for matched pairs
for t in range(NF): # score detection against GT
    for (gx, gy, g, v) in GT[t]: # each GT entity
        if not v: # skip occluded ones (nothing to detect)
            continue # not visible -> excluded from recall
        vis += 1 # one more visible entity
        best = min([abs(d[0]-gx)+abs(d[1]-gy) for d in ALLD[t] if d[2] == g] or [1e9]) # closest same-group detection (L1)
        if best < 6: # within 6px counts as found
            hits += 1 # detection hit
            errs.append(best) # record error
print(f'detection recall={hits}/{vis}={hits/max(vis,1):.3f} mean_err={np.mean(errs):.2f}px') # headline quality
assert hits/max(vis, 1) > 0.90 # fail loudly if the detector is broken

In [ ]:
GATE, MAX_MISS = 12.0, 3 # max match distance (px) and frames a track coasts while hidden
tracks, nxt = {}, 0 # tracks: id -> dict(group, pos, misses, hist); nxt = next free id
for t in range(NF): # greedy per-group centroid matching, frame by frame
    for g in ('A', 'B', 'ball'): # match groups independently (color already disambiguates)
        ds = [d for d in ALLD[t] if d[2] == g] # detections of this group
        act = [i for i, tr in tracks.items() if tr['group'] == g and tr['misses'] <= MAX_MISS] # live tracks
        md = {} # track id -> matched detection index
        used = set() # detection indices already consumed
        for i in act: # each live track picks its nearest free detection
            bi, bd = -1, GATE # best detection index and distance
            for j, d in enumerate(ds): # scan free detections
                if j in used: # already taken
                    continue # skip
                dd = abs(d[0]-tracks[i]['pos'][0]) + abs(d[1]-tracks[i]['pos'][1]) # L1 distance
                if dd < bd: # closer match
                    bi, bd = j, dd # adopt
            if bi >= 0: # matched
                md[i] = bi # record match
                used.add(bi) # consume detection
        for i in act: # update every live track
            if i in md: # matched: snap to detection
                tracks[i]['pos'] = (ds[md[i]][0], ds[md[i]][1]) # new centroid
                tracks[i]['misses'] = 0 # reset miss counter
                tracks[i]['hist'].append((t,) + tracks[i]['pos'] + (False,)) # log real observation
            else: # unmatched: coast (keep last pos, grow misses)
                tracks[i]['misses'] += 1 # one more missed frame
                tracks[i]['hist'].append((t,) + tracks[i]['pos'] + (True,)) # log coasted point
        for j, d in enumerate(ds): # leftover detections start new tracks
            if j not in used: # unmatched detection
                tracks[nxt] = {'group': g, 'pos': (d[0], d[1]), 'misses': 0, 'hist': [(t, d[0], d[1], False)]} # birth
                nxt += 1 # consume the id
n_gt = NPA+NPB+1 # true entity count (8 players + ball)
print(f'tracking: {len(tracks)} ids created for {n_gt} entities') # id count vs truth
own = {} # GT-entity-index -> majority track id vote
for gi in range(n_gt): # vote: which track saw this entity most
    votes = {} # track id -> frames observed near this entity
    for t in range(NF): # scan frames
        gx, gy, g, v = GT[t][gi] # GT position
        if not v: # hidden: no vote
            continue # skip
        for i, tr in tracks.items(): # find nearby track points
            if tr['group'] != g: # wrong group cannot match
                continue # skip
            for (ft, px, py, co) in tr['hist']: # scan track history
                if ft == t and abs(px-gx)+abs(py-gy) < 6 and not co: # real observation near GT
                    votes[i] = votes.get(i, 0)+1 # one vote
    own[gi] = max(votes, key=votes.get) if votes else -1 # majority track or -1
sw = sum(1 for gi in range(n_gt) for t in range(NF) if GT[t][gi][3] and not any(ft == t and abs(px-GT[t][gi][0])+abs(py-GT[t][gi][1]) < 6 and not co for (ft, px, py, co) in tracks.get(own[gi], {'hist': []})['hist'])) # frames where the majority track is absent
print(f'id coverage gaps: {sw} entity-frames (occlusions/coasts explain most)') # headline continuity
assert sw < 60 # fail loudly if tracking falls apart (6% occlusion budget => far fewer)

In [ ]:
SRC_P = np.array([[IN[0], IN[1]], [IN[2], IN[1]], [IN[2], IN[3]], [IN[0], IN[3]]], dtype=np.float64) # image corners of the pitch rect
DST_P = np.array([[0, 0], [PITCH_W, 0], [PITCH_W, PITCH_H], [0, PITCH_H]], dtype=np.float64) # same corners in meters
def homography_dlt(src, dst): # solve H (3x3) with the direct linear transform
    A = [] # 2 rows per correspondence: Ah = 0
    for (x, y), (X, Y) in zip(src, dst): # each point pair adds two equations
        A.append([-x, -y, -1, 0, 0, 0, X*x, X*y, X]) # row for the X equation
        A.append([0, 0, 0, -x, -y, -1, Y*x, Y*y, Y]) # row for the Y equation
    _, _, Vt = np.linalg.svd(np.array(A)) # null space = last row of V^T
    Hm = Vt[-1].reshape(3, 3) # reshape 9-vector to 3x3
    return Hm / Hm[2, 2] # normalize so H[2,2] = 1
def apply_H(Hm, pts): # map Nx2 image points to pitch meters
    p = np.asarray(pts, dtype=np.float64) # ensure array
    h = (Hm @ np.concatenate([p, np.ones((len(p), 1))], axis=1).T).T # homogeneous multiply
    return h[:, :2] / h[:, 2:3] # divide by w -> meters
Hm = homography_dlt(SRC_P, DST_P) # solve once from the 4 known corners
reproj = np.abs(apply_H(Hm, SRC_P) - DST_P).max() # max corner reprojection error
print('homography max corner err (m):', reproj) # must be ~0
assert reproj < 1e-6 # fail loudly if DLT is wrong
rows = [] # tracks.csv rows: frame,id,group,x_img,y_img,x_m,y_m,coasted
for i, tr in tracks.items(): # export every track point in both coordinate frames
    pm = apply_H(Hm, [(h[1], h[2]) for h in tr['hist']]) # map this track to meters
    for k, h in enumerate(tr['hist']): # h = (frame, x_img, y_img, coasted)
        rows.append([h[0], i, tr['group'], round(h[1], 2), round(h[2], 2), round(float(pm[k, 0]), 3), round(float(pm[k, 1]), 3), int(h[3])]) # one row per point
with open('./results/tracks.csv', 'w', newline='') as f: # save per-frame tracks
    w = csv.writer(f) # need writer
    w.writerow(['frame', 'id', 'group', 'x_img', 'y_img', 'x_m', 'y_m', 'coasted']) # header
    w.writerows(sorted(rows)) # write sorted by frame
summ = [] # summary.csv rows: per-id distance + top speed
for i, tr in tracks.items(): # one summary row per track id
    pm = apply_H(Hm, [(h[1], h[2]) for h in tr['hist']]) # meters trajectory
    step = np.linalg.norm(pm[1:]-pm[:-1], axis=1) # per-frame step lengths in meters
    dist = float(step.sum()) # total distance run
    top = float(step.max()*FPS) if len(step) else 0.0 # top speed in m/s
    summ.append([i, tr['group'], len(tr['hist']), round(dist, 2), round(top, 2)]) # id summary
with open('./results/summary.csv', 'w', newline='') as f: # save summary
    w = csv.writer(f) # need writer
    w.writerow(['id', 'group', 'frames', 'dist_m', 'top_speed_ms']) # header
    w.writerows(summ) # write rows
for r in sorted(summ)[:10]: # print first 10 track summaries
    print('track', r) # id, group, frames, distance, top speed

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4)) # 3 panels: frame, pitch tracks, heatmap
ax[0].imshow(frames[NF//2]) # middle frame as RGB
for d in ALLD[NF//2]: # overlay its detections
    c = 'red' if d[2] == 'A' else ('cyan' if d[2] == 'B' else 'yellow') # color per group
    ax[0].add_patch(patches.Rectangle((d[3], d[4]), d[5]-d[3], d[6]-d[4], fill=False, edgecolor=c, linewidth=1.5)) # box overlay
ax[0].set_title(f'frame {NF//2} detections') # panel title
ax[0].axis('off') # hide axes
ax[1].set_xlim(-1, PITCH_W+1) # pitch x range in meters
ax[1].set_ylim(-1, PITCH_H+1) # pitch y range in meters
ax[1].add_patch(patches.Rectangle((0, 0), PITCH_W, PITCH_H, fill=False, edgecolor='white')) # pitch outline
ax[1].set_facecolor('green') # green background
allm = [] # every tracked point in meters (for tracks + heatmap)
for i, tr in tracks.items(): # draw each trajectory
    pm = apply_H(Hm, [(h[1], h[2]) for h in tr['hist']]) # map to meters
    allm.append(pm) # collect for the heatmap
    ax[1].plot(pm[:, 0], pm[:, 1], linewidth=1.2, label=f"id{i}") # trajectory line
ax[1].set_title('trajectories (meters)') # panel title
ax[1].legend(fontsize=6, ncol=3) # compact legend
allm = np.concatenate(allm) # stack all points
ax[2].hist2d(allm[:, 0], allm[:, 1], bins=[16, 12], range=[[0, PITCH_W], [0, PITCH_H]]) # 2D position histogram
ax[2].set_title('position heatmap') # panel title
ax[2].set_xlabel('x (m)') # x label
ax[2].set_ylabel('y (m)') # y label
fig.suptitle('P9 sports vision: detect->track->homography') # figure title
fig.savefig('./results/pipeline.png', dpi=100) # save the figure
plt.close(fig) # free memory
print('done: see ./results/tracks.csv, summary.csv, pipeline.png') # artifact pointer

## What to write in your README (7 questions)
1. What did I build? 2. How do detection / tracking / homography work?
3. What recall / ms-per-frame / distances? 4. What surprised you? 5. Bottleneck?
6. What if GATE / occlusion / missed frames change? 7. What next (YOLO, Kalman, real video)?